Showcase the categorization of customer feedback using fine-tuned GPT-2.

We have a dataset of customer feedback that we want to categorize into different categories. We will use the fine-tuned GPT-2 model to classify the feedback into different categories.

In [1]:
# Load datasets
from datasets import load_dataset

from pathlib import Path

notebook_path = Path.cwd()

# Save synthetic data to CSV
train_path = notebook_path / "synthetic_train_data.csv"
test_path = notebook_path / "synthetic_test_data.csv"

dataset = load_dataset("csv", data_files={"train": str(train_path.absolute()), "test": str(test_path.absolute())})

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

We need to load the fine-tuned GPT-2 model and the dataset of customer feedback. We will then use the model to predict the category of each feedback.

We start this by loading the tokenizer and the model.

In [2]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load the tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Add a padding token to the tokenizer
tokenizer.add_special_tokens({"pad_token": "[PAD]"})
model.resize_token_embeddings(len(tokenizer))


# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


# Tokenize the training and testing datasets
tokenized_datasets = dataset.map(tokenize_function, batched=True)

/home/amruthvvkp/projects/ai-playground/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Now we need to fine-tune the model on the dataset of customer feedback. We will use the fine-tuned model to predict the category of each feedback.

We will then evaluate the performance of the model by comparing the predicted categories with the actual categories of the feedback.

Finally, we will use the model to predict the category of new customer feedback.

In [3]:
import numpy as np
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Fine-tune the model
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=9,
    weight_decay=0.01,
)

# Use a data collator for causal language modeling (not MLM, as GPT-2 is autoregressive)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Turn off MLM as this is a causal language model
)

# Ensure that labels are passed for loss calculation
# Ensure that labels are passed for loss calculation
# The issue with the accuracy being 0 is likely due to the way the accuracy is being calculated for a causal language model like GPT-2. In causal language modeling, the model predicts the next token in a sequence, and the labels are shifted versions of the input sequences. Therefore, calculating accuracy as the proportion of correct predictions over the entire sequence might not be appropriate.
# Instead, you should calculate the accuracy token-by-token, comparing each predicted token with the corresponding label token.
# Here's how you can modify the compute_metrics function to calculate token-level accuracy:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Flatten the predictions and labels to calculate token-level accuracy
    flattened_predictions = predictions.flatten()
    flattened_labels = labels.flatten()

    # Mask out the padding tokens (assuming padding token ID is -100)
    mask = flattened_labels != -100
    correct_predictions = flattened_predictions[mask] == flattened_labels[mask]

    accuracy = correct_predictions.mean()
    return {"accuracy": accuracy}


# Use the Trainer for training with labels (causal language modeling)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,  # Pass the causal language modeling collator
    compute_metrics=compute_metrics,
)

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,87.779984,0.000000


KeyboardInterrupt: 

We need to define a preprocessing function to clean the text data and convert it into a format that the model can understand. We also need to define a function to predict the category of the feedback using the fine-tuned GPT-2 model.

We are trying to use Langchain to predict the category of customer feedback using fine-tuned GPT-2. We will use the fine-tuned model to classify the feedback into different categories.

In [3]:
# Custom Asynchronous Transformation Step
from langchain_core.runnables.base import Runnable
from langchain_core.runnables.utils import Input, Output
from langchain_core.runnables.config import RunnableConfig
from typing import AsyncIterator, Optional, Any


class CustomAsyncTransformStep(Runnable[Input, Output]):
    tokenizer = tokenizer

    async def atransform(
        self, input: AsyncIterator[Input], config: Optional[RunnableConfig] = None, **kwargs: Any
    ) -> AsyncIterator[Output]:
        # Implement your custom asynchronous transformation logic here
        async for item in input:
            # Example transformation: clean and tokenize text
            cleaned_text = self.clean_text(item)
            tokens = self.tokenize(cleaned_text)
            yield Output(data=" ".join(tokens))

    def clean_text(self, text: str) -> str:
        # Perform general cleaning like lowercasing, punctuation removal, etc.
        cleaned_text = text.lower().replace(".", "").replace(",", "").replace("!", "").replace("?", "")
        return cleaned_text

    def tokenize(self, text: str) -> list[str]:
        # Tokenize the text using the tokenizer
        tokens = self.tokenizer.tokenize(text)
        return tokens

In [5]:
import pandas as pd
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from sklearn.preprocessing import MinMaxScaler
from langchain.tools import BaseTool
from typing import Optional, Union, List
from langchain.callbacks.manager import CallbackManagerForToolRun, AsyncCallbackManagerForToolRun
from transformers import PreTrainedModel


class DataPreprocessingTool(BaseTool, CustomAsyncTransformStep):
    name: str = "DataPreprocessingTool"
    description: str = "A tool for preprocessing and structuring unstructured data."
    tokenizer: GPT2Tokenizer = tokenizer
    model: PreTrainedModel = model

    def _run(
        self,
        unstructured_data: Union[str, List[str]],
        run_manager: Optional[Union[CallbackManagerForToolRun, AsyncCallbackManagerForToolRun]] = None,
    ) -> pd.DataFrame:
        # Main function to orchestrate the preprocessing and structuring steps
        structured_data = self.structure_data(unstructured_data)
        normalized_data = self.normalize_data(structured_data)
        enriched_data = self.enrich_data(normalized_data)
        return enriched_data

    async def _arun(
        self, unstructured_data: Union[str, List[str]], run_manager: Optional[AsyncCallbackManagerForToolRun] = None
    ) -> pd.DataFrame:
        # Asynchronous version of the _run method
        return self._run(unstructured_data, run_manager)

    def structure_data(self, data: Union[str, List[str]]) -> pd.DataFrame:
        # Transforms unstructured data into a structured format
        structured_data_list = []
        for item in data:
            cleaned_text = self.clean_text(item)
            tokens = self.tokenize(cleaned_text)
            vector = self.vectorize_text(cleaned_text)
            structured_data_list.append({"cleaned_text": cleaned_text, "tokens": tokens, "vector": vector})
        structured_data = pd.DataFrame(structured_data_list)
        return structured_data

    def normalize_data(self, data: pd.DataFrame) -> pd.DataFrame:
        # Normalizes data (e.g., scaling features to a standard range)
        scaler = MinMaxScaler()
        data["vector"] = list(scaler.fit_transform(data["vector"].tolist()))
        return data

    def enrich_data(self, data: pd.DataFrame) -> pd.DataFrame:
        # Enriches data with additional information
        data["num_tokens"] = data["tokens"].apply(len)
        return data

    def vectorize_text(self, text: str) -> List[float]:
        # Converts text to a numerical vector using the tokenizer
        inputs = self.tokenizer(text, return_tensors="pt")
        vector = inputs["input_ids"][0].tolist()
        return vector

    def categorize_feedback(self, feedback: str) -> str:
        # Preprocess the feedback
        preprocessed_feedback = self.clean_text(feedback)
        # Tokenize the preprocessed feedback for the model
        inputs = self.tokenizer(preprocessed_feedback, return_tensors="pt")
        # Generate category using the fine-tuned model
        outputs = self.model.generate(inputs["input_ids"], max_length=50)
        category = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return category


# Example usage
preprocessing_tool = DataPreprocessingTool()
user_feedback = "The app crashes when I try to upload a photo."
category = preprocessing_tool.categorize_feedback(user_feedback)
print(f"Categorized as: {category}")

/home/amruthvvkp/projects/ai-playground/.venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:172: UserWarning: Field name "tokenizer" in "DataPreprocessingTool" shadows an attribute in parent "CustomAsyncTransformStep"
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Categorized as: the app crashes when i try to upload a photo
